# Alpha158 因子构建与模型数据准备

直接读取本项目 `data_loader.ipynb` 下载的 Tushare CSV，不调用下载接口。
本 notebook 输出 **158 个日频量价因子**、未来收益标签和时间切分标记，供后续模型共用。

**使用顺序**：等待下载结束 → 核对配置 → 从上到下运行。数据未齐时仍可运行公式自检，
全量构建会明确跳过。原始行情、下载 notebook 均不修改。

因子名称、顺序和公式以 [Qlib Alpha158DL](https://github.com/microsoft/qlib/blob/main/qlib/contrib/data/loader.py)
为依据；滚动细节参照 [Qlib operators](https://github.com/microsoft/qlib/blob/main/qlib/data/ops.py)。
结构为 9 个 K 线因子 + 4 个当日价格比率 + 29 类滚动因子 × 5 个窗口 = 158。
此处用 pandas / NumPy 本地计算，不要求安装 Qlib。

**适用时点**：t 日收盘后产生特征。默认标签为 `close[t+2] / close[t+1] - 1`，
与 [Qlib Alpha158 默认标签](https://github.com/microsoft/qlib/blob/main/qlib/contrib/data/handler.py) 一致。
它是研究用的收盘到收盘收益，实际交易测试仍需另行考虑成交时点、涨跌停、停牌及费用。

## 1. 环境和配置

必需依赖：`numpy>=1.24`、`pandas>=1.5`；可选 `pyarrow`，用于更小、更快的 Parquet 输出。
若缺少依赖，请在当前 notebook 内核环境安装：`%pip install numpy pandas pyarrow`。
未安装 pyarrow 时自动使用 gzip CSV。

日期默认对应下载计划 2022–2025；前 61 个有效观测用于预热。
调整 `START_DATE / END_DATE` 时，请同时检查训练、验证、测试边界。
股票范围默认读取下载时保存的 `stock_codes_a_share.csv`（本项目为沪深 A 股），包含退市股票的历史记录；不按当前上市状态筛选。
同时依据 `stock_basic.csv` 的 `list_date` 排除上市前记录。指定 `INSTRUMENTS` 时需为该股票池内的子集。
若研究范围变更，应先统一更新股票池与基本信息，再重建所有模块的数据。

In [ ]:
from pathlib import Path
import importlib.util
import json
import shutil
import uuid
import warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 12)
PROJECT_DIR = Path.cwd()
# 在项目目录或其子目录启动内核均可；其他位置启动时回退到本项目的绝对路径。
PROJECT_DIR = next((p for p in [PROJECT_DIR, *PROJECT_DIR.parents]
                    if (p / 'data' / 'raw' / 'daily').is_dir()),
                   Path(r'E:\document-prefer\python for finanace\multi-model-bactest'))
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
OUTPUT_ROOT = PROJECT_DIR / 'data' / 'processed' / 'alpha158'

START_DATE = '20220101'
END_DATE = '20251231'
TRAIN_END = '2023-12-31'
VALID_END = '2024-12-31'
TEST_END = '2025-12-31'
# 默认沿用下载时保存的股票池，包含退市股票的历史记录。
UNIVERSE_FILE = RAW_DIR / 'basic' / 'stock_codes_a_share.csv'
STOCK_BASIC_FILE = RAW_DIR / 'basic' / 'stock_basic.csv'
INSTRUMENTS = None       # None = 上述股票池；指定列表时仅取池内子集
INVALID_ROW_POLICY = 'quarantine'  # 'quarantine' 隔离异常行；'raise' 严格报错
MAX_QUARANTINE_FRACTION = 0.001    # 池内且已上市行，全期间异常比例上限 0.1%
MAX_BAD_DAY_FRACTION = 0.02        # 单个交易日异常比例上限 2%
LABEL_HORIZONS = (1, 5, 10)  # LABEL0 为 1 日；另提供 LABEL5、LABEL10
LABEL_ENTRY_LAG = 1      # t+1 收盘进入，t+1+h 收盘退出
MIN_HISTORY = 61         # ROC60 至少需要 61 个观测
ADJUST_VOLUME = True     # 股数按价格复权倍率反向调整，保持量价口径一致
BATCH_SIZE = 100         # 分股票批次落盘，避免一次持有全市场 158 列
RUN_BUILD = True         # 数据不齐自动跳过；改为 False 可只执行预检、自检
OUTPUT_FORMAT = 'parquet' if importlib.util.find_spec('pyarrow') else 'csv.gz'
EPS = 1e-12
WINDOWS = (5, 10, 20, 30, 60)
print(f'项目：{PROJECT_DIR}\n输出格式：{OUTPUT_FORMAT}')

## 2. 下载覆盖检查

必需数据是 `daily/YYYY/YYYYMMDD.csv`、`adj_factor/YYYY/YYYYMMDD.csv` 和交易日历。
`daily_basic` 的估值、市值、换手率不属于标准 Alpha158，因此不拼入这 158 列。
文件覆盖缺失时等待下载完成。文件齐全后的少量行级异常，由第 3 节隔离并明确报告；大范围缺失仍阻止构建。
预检只查看覆盖及空文件，真正计算前还会检查 CSV 字段、数值和主键。

In [ ]:
def date_files(folder):
    files = {}
    for path in sorted(folder.rglob('*.csv')):
        if len(path.stem) == 8 and path.stem.isdigit():
            if path.stem in files:
                raise ValueError(f'同一日期存在多个文件：{path.stem}，目录 {folder}')
            files[path.stem] = path
    return files

def read_calendar(raw_dir, start, end):
    folder = raw_dir / 'basic'
    files = sorted(folder.glob('trade_dates_*.csv'))
    if not files:
        raise FileNotFoundError(f'未找到 {folder}/trade_dates_*.csv')
    values = []
    for path in files:
        frame = pd.read_csv(path, dtype=str)
        if 'trade_date' not in frame:
            raise ValueError(f'{path} 缺少 trade_date')
        values.extend(frame['trade_date'].tolist())
    cal = pd.DatetimeIndex(pd.to_datetime(values, format='%Y%m%d', errors='raise'))
    cal = cal.unique().sort_values()
    cal = cal[(cal >= pd.Timestamp(start)) & (cal <= pd.Timestamp(end))]
    if cal.empty:
        raise ValueError('配置日期范围内没有交易日')
    return cal

def check_coverage(raw_dir, start, end):
    calendar = read_calendar(raw_dir, start, end)
    expected = set(calendar.strftime('%Y%m%d'))
    manifests, rows = {}, []
    for group in ('daily', 'adj_factor'):
        files = date_files(raw_dir / group)
        found = {day: path for day, path in files.items() if day in expected}
        empty = [day for day, path in found.items() if path.stat().st_size == 0]
        missing = sorted(expected - set(found))
        manifests[group] = found
        rows.append({'dataset': group, 'expected_days': len(expected),
                     'found_days': len(found), 'missing_days': len(missing),
                     'empty_files': len(empty), 'first_missing': missing[:5]})
    report = pd.DataFrame(rows).set_index('dataset')
    ready = bool((report[['missing_days', 'empty_files']] == 0).all().all())
    return calendar, manifests, report, ready

calendar, manifests, coverage, data_ready = check_coverage(RAW_DIR, START_DATE, END_DATE)
display(coverage)
print('覆盖检查通过。' if data_ready else '下载尚未覆盖全部必需交易日；全量构建暂时跳过。下载结束后重新运行本单元。')

## 3. 读取及复权

[Tushare 日线单位](https://tushare.pro/document/1?doc_id=27)：`vol` 为手、`amount` 为千元。
因而原始 VWAP = `amount × 1000 / (vol × 100)`，不能直接用 `amount/vol`。
OHLC 和 VWAP 同乘 `adj_factor / 首个有效 adj_factor`，使用历史起点锚定，避免依赖未来的复权基准。
默认成交量换算为股，并除以同一倍率；这是本项目的输入约定。
公式遵循 Qlib，原始供应商、量纲、停牌和复权基准可能与 Qlib 官方二进制数据不同，不承诺逐位一致。

每只股票按市场交易日历补索引，停牌日保留 NaN，再计算滚动窗口和标签。
因此标签不会把复牌日误当成“下一交易日”，也不会跨股票滚动。

**输入质量处理**：先应用股票池和上市日期，再合并复权因子。默认隔离少量缺失/无效记录，
在 `data/checks/alpha158_input/` 保存逐日统计和排除明细，成功运行目录中也保存一份。
隔离比例上限默认全期间 0.1%、单日 2%；超限停止。`INVALID_ROW_POLICY="raise"` 可改为严格模式。
被隔离行在日历上仍是 NaN，不前填、不后填、不把复权因子设为 1；有效退市历史记录照常保留。

In [ ]:
DAILY_COLUMNS = ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'vol', 'amount']
KEYS = ['ts_code', 'trade_date']

def load_universe(instruments=None, universe_path=UNIVERSE_FILE, basic_path=STOCK_BASIC_FILE):
    universe_path, basic_path = Path(universe_path), Path(basic_path)
    pool = pd.read_csv(universe_path, usecols=['ts_code'], dtype=str)
    basic = pd.read_csv(basic_path, usecols=['ts_code', 'list_date'], dtype=str)
    if pool.empty or pool.ts_code.isna().any() or pool.ts_code.duplicated().any():
        raise ValueError(f'股票池为空、代码缺失或重复：{universe_path}')
    if basic.ts_code.isna().any() or basic.ts_code.duplicated().any():
        raise ValueError(f'股票基本信息代码缺失或重复：{basic_path}')
    basic['list_date'] = pd.to_datetime(basic.list_date, format='%Y%m%d', errors='raise')
    selected = set(pool.ts_code)
    if instruments is not None:
        if isinstance(instruments, str) or not len(instruments):
            raise ValueError('INSTRUMENTS 需为非空股票代码列表，或 None')
        requested = set(instruments)
        unknown = requested - selected
        if unknown:
            raise ValueError(f'所选代码不在下载股票池中：{sorted(unknown)[:5]}；请先统一股票池及基本信息')
        selected = requested
    listing = basic.set_index('ts_code').list_date.reindex(sorted(selected))
    if listing.isna().any():
        raise ValueError(f'股票池内缺少上市日期：{listing.index[listing.isna()].tolist()[:5]}')
    info = {'universe_file': str(universe_path.resolve()),
            'universe_sha256': __import__('hashlib').sha256(universe_path.read_bytes()).hexdigest(),
            'stock_basic_file': str(basic_path.resolve()),
            'stock_basic_sha256': __import__('hashlib').sha256(basic_path.read_bytes()).hexdigest(),
            'selected_codes': sorted(selected), 'selected_code_count': len(selected)}
    return sorted(selected), listing, info

def read_daily_file(path, columns, expected_day=None, instruments=None):
    frame = pd.read_csv(path, usecols=columns, dtype={'ts_code': str, 'trade_date': str})
    rows_read = len(frame)
    if frame.empty:
        raise ValueError(f'文件无数据（可能正在写入）：{path}')
    if frame[KEYS].isna().any().any():
        raise ValueError(f'日期或股票代码为空：{path}')
    if expected_day and not frame['trade_date'].eq(expected_day).all():
        raise ValueError(f'文件日期与内容不符：{path}')
    if frame.duplicated(KEYS).any():
        raise ValueError(f'重复 (ts_code, trade_date)：{path}')
    if instruments is not None:
        frame = frame[frame.ts_code.isin(instruments)].copy()
    for column in set(columns) - set(KEYS):
        frame[column] = pd.to_numeric(frame[column], errors='raise')
    frame['trade_date'] = pd.to_datetime(frame['trade_date'], format='%Y%m%d')
    frame.attrs['rows_read'] = rows_read
    return frame

def validate_merged_rows(frame):
    # 缺失复权记录与无效数值分开标记；不插值、不前填、不填 1。
    numeric = frame[DAILY_COLUMNS[2:] + ['adj_factor']]
    flags = pd.DataFrame(index=frame.index)
    flags['missing_adj_record'] = frame['_merge'].eq('left_only')
    flags['nonfinite_daily'] = ~np.isfinite(numeric[DAILY_COLUMNS[2:]].to_numpy(dtype=float)).all(axis=1)
    flags['invalid_adj_factor'] = frame['_merge'].eq('both') & (
        ~np.isfinite(frame.adj_factor.to_numpy(dtype=float)) | frame.adj_factor.le(0))
    flags['nonpositive_price'] = frame[['open', 'high', 'low', 'close']].le(0).any(axis=1)
    flags['negative_volume_amount'] = frame[['vol', 'amount']].lt(0).any(axis=1)
    flags['invalid_ohlc'] = ((frame.high < frame[['open', 'close', 'low']].max(axis=1)) |
                             (frame.low > frame[['open', 'close', 'high']].min(axis=1)))
    flags['inconsistent_volume_amount'] = ((frame.vol.eq(0) & frame.amount.gt(0)) |
                                            (frame.vol.gt(0) & frame.amount.eq(0)))
    return flags

def load_daily_panel(manifests, instruments=None, universe_path=UNIVERSE_FILE,
                     basic_path=STOCK_BASIC_FILE, invalid_row_policy=INVALID_ROW_POLICY,
                     max_bad_fraction=MAX_QUARANTINE_FRACTION,
                     max_bad_day_fraction=MAX_BAD_DAY_FRACTION, audit_dir=None):
    if invalid_row_policy not in ('quarantine', 'raise'):
        raise ValueError('invalid_row_policy 必须为 quarantine 或 raise')
    if not 0 <= max_bad_fraction <= 1 or not 0 <= max_bad_day_fraction <= 1:
        raise ValueError('异常比例阈值需位于 [0,1]')
    selected, listing, universe = load_universe(instruments, universe_path, basic_path)
    parts, audits, excluded = [], [], []
    days = sorted(manifests['daily'])
    candidates = quarantined = 0
    bad_days = []
    for number, day in enumerate(days, 1):
        daily = read_daily_file(manifests['daily'][day], DAILY_COLUMNS, day, selected)
        source_rows = daily.attrs['rows_read']
        daily['list_date'] = daily.ts_code.map(listing)
        prelisting = daily.trade_date.lt(daily.list_date)
        if prelisting.any():
            removed = daily.loc[prelisting].copy()
            removed['adj_factor'] = np.nan
            removed['reason'] = 'before_listing'
            excluded.append(removed)
        frame = daily.loc[~prelisting].copy()
        adjustment = read_daily_file(manifests['adj_factor'][day], KEYS + ['adj_factor'], day, selected)
        frame = frame.merge(adjustment, on=KEYS, how='left', validate='one_to_one', indicator=True)
        flags = validate_merged_rows(frame)
        bad = flags.any(axis=1)
        count, bad_count = len(frame), int(bad.sum())
        candidates += count
        quarantined += bad_count
        if bad_count:
            removed = frame.loc[bad].drop(columns='_merge').copy()
            removed['reason'] = [','.join(flags.columns[row]) for row in flags.loc[bad].to_numpy()]
            excluded.append(removed)
        if count and bad_count / count > max_bad_day_fraction:
            bad_days.append(day)
        audits.append({'trade_date': day, 'raw_daily_rows': source_rows,
                       'outside_universe': source_rows - len(daily),
                       'before_listing': int(prelisting.sum()), 'candidate_rows': count,
                       'quarantined_rows': bad_count, 'retained_rows': count-bad_count})
        parts.append(frame.loc[~bad].drop(columns=['_merge', 'list_date']))
        if number % 200 == 0 or number == len(days):
            print(f'读取交易日 {number}/{len(days)}')
    coverage = pd.DataFrame(audits)
    excluded_rows = pd.concat(excluded, ignore_index=True) if excluded else pd.DataFrame(
        columns=DAILY_COLUMNS + ['list_date', 'adj_factor', 'reason'])
    report_path = None
    if audit_dir is not None:
        audit_dir = Path(audit_dir)
        audit_dir.mkdir(parents=True, exist_ok=True)
        coverage.to_csv(audit_dir / 'input_coverage.csv', index=False)
        excluded_rows.to_csv(audit_dir / 'excluded_input_rows.csv', index=False)
        report_path = audit_dir / 'excluded_input_rows.csv'
    bad_fraction = quarantined / candidates if candidates else 0.
    # 大范围缺失或严格模式仍阻止构建；异常明细在报错前已落盘。
    if quarantined and (invalid_row_policy == 'raise' or bad_fraction > max_bad_fraction or bad_days):
        examples = excluded_rows.loc[excluded_rows.reason.ne('before_listing'), ['ts_code', 'trade_date', 'reason']].head(5)
        raise ValueError(f'池内已上市行情异常 {quarantined}/{candidates} ({bad_fraction:.4%})；'
                         f'超限日期 {bad_days[:5]}；明细 {report_path}。示例：\n{examples.to_string(index=False)}')
    if quarantined or not excluded_rows.empty:
        warnings.warn(f'已按上市日期剔除 {int(coverage.before_listing.sum())} 行；'
                      f'隔离 {quarantined} 条异常行（{bad_fraction:.4%}）。'
                      f'复权因子未填补；详情：{report_path or "返回 panel.attrs[excluded_input_rows]"}',
                      UserWarning)
    panel = pd.concat(parts, ignore_index=True).sort_values(KEYS, kind='stable')
    if panel.empty:
        raise ValueError('股票池、上市日期和异常记录过滤后没有可用行情')
    # 被隔离的日期在 prepare_instrument 中重新索引为 NaN，不压缩交易日。
    panel.attrs['input_coverage'] = coverage
    panel.attrs['excluded_input_rows'] = excluded_rows
    panel.attrs['universe'] = universe
    panel.attrs['input_quality'] = {'invalid_row_policy': invalid_row_policy,
        'max_bad_fraction': max_bad_fraction, 'max_bad_day_fraction': max_bad_day_fraction,
        'candidate_rows': candidates, 'quarantined_rows': quarantined,
        'before_listing_rows': int(coverage.before_listing.sum()),
        'outside_universe_rows': int(coverage.outside_universe.sum())}
    return panel

def prepare_instrument(raw, calendar, adjust_volume=True):
    raw = raw.copy(deep=False)
    raw.attrs = {}  # 输入审计仅属于面板，不向每只股票的滚动计算传播
    raw = raw.sort_values('trade_date').set_index('trade_date')
    if raw.index.has_duplicates:
        raise ValueError('单只股票日期重复')
    bars = raw.reindex(calendar).copy()
    bars.index.name = 'datetime'
    bars['observed'] = bars['close'].notna()
    bars['history_count'] = bars.observed.cumsum()
    anchor = raw.adj_factor.dropna().iloc[0]
    ratio = bars.adj_factor / anchor
    shares = bars.vol * 100.0
    bars['vwap'] = (bars.amount * 1000.0 / shares.where(shares > 0)) * ratio
    for field in ('open', 'high', 'low', 'close'):
        bars[field] = bars[field] * ratio
    bars['volume'] = shares / ratio if adjust_volume else shares
    bars['tradeable_observed'] = bars.observed & shares.gt(0)
    return bars

## 4. Alpha158 计算

滚动窗口为 5、10、20、30、60 个市场交易日，仅使用当日及过去的数据。
对齐 Qlib 的 `min_periods=1`、样本标准差 `ddof=1`、平均并列排名，
以及极值第一次出现的 **从 1 开始的位置**。`ROC` 是过去收盘价/当前收盘价，
`BETA` 是价格对时间的回归斜率/当前收盘价，并非市场 beta。
常数序列的相关系数和 R² 保留 NaN；不反向填充，也不提前填零。

In [ ]:
KBAR_NAMES = ['KMID', 'KLEN', 'KMID2', 'KUP', 'KUP2', 'KLOW', 'KLOW2', 'KSFT', 'KSFT2']
ROLLING_NAMES = ['ROC', 'MA', 'STD', 'BETA', 'RSQR', 'RESI', 'MAX', 'MIN',
                 'QTLU', 'QTLD', 'RANK', 'RSV', 'IMAX', 'IMIN', 'IMXD',
                 'CORR', 'CORD', 'CNTP', 'CNTN', 'CNTD', 'SUMP', 'SUMN',
                 'SUMD', 'VMA', 'VSTD', 'WVMA', 'VSUMP', 'VSUMN', 'VSUMD']
FEATURE_NAMES = KBAR_NAMES + ['OPEN0', 'HIGH0', 'LOW0', 'VWAP0'] + [
    f'{name}{window}' for name in ROLLING_NAMES for window in WINDOWS]
assert len(FEATURE_NAMES) == len(set(FEATURE_NAMES)) == 158

def rolling_regression(series, window):
    # 用滚动矩统计回归，避免逐窗口 Python 回调；缺失日期的时间位置不压缩。
    x = pd.Series(np.arange(len(series), dtype=float), index=series.index)
    valid_x = x.where(series.notna())
    def mean(value):
        return value.rolling(window, min_periods=2).mean()
    mx, my = mean(valid_x), mean(series)
    covariance = mean(valid_x * series) - mx * my
    vx = mean(valid_x**2) - mx**2
    vy = mean(series**2) - my**2
    slope = covariance / vx.where(vx > 0)
    r2 = (covariance**2 / (vx * vy).where((vx > 0) & (vy > 0))).clip(0, 1)
    r2 = r2.mask(np.isclose(series.rolling(window, min_periods=1).std(), 0, atol=2e-5))
    residual = series - (my + slope * (x - mx))
    return slope, r2, residual

def qlib_corr(left, right, window):
    left, right = left.replace([np.inf, -np.inf], np.nan), right.replace([np.inf, -np.inf], np.nan)
    a, b = left.rolling(window, min_periods=1), right.rolling(window, min_periods=1)
    result = a.corr(right)
    constant = np.isclose(a.std(), 0, atol=2e-5) | np.isclose(b.std(), 0, atol=2e-5)
    return result.mask(constant)

def alpha158(bars):
    o, h, l, c, v = (bars[name].astype(float) for name in ('open', 'high', 'low', 'close', 'volume'))
    spread = h - l
    upper, lower = h - np.maximum(o, c), np.minimum(o, c) - l
    shifted = 2*c - h - l
    values = [ (c-o)/o, spread/o, (c-o)/(spread+EPS), upper/o,
               upper/(spread+EPS), lower/o, lower/(spread+EPS), shifted/o, shifted/(spread+EPS)]
    out = dict(zip(KBAR_NAMES, values))
    out.update(OPEN0=o/c, HIGH0=h/c, LOW0=l/c, VWAP0=bars.vwap/c)
    previous = c.shift(1)
    price_ratio = c / previous
    price_delta, volume_delta = c.diff(), v.diff()
    log_volume = np.log1p(v)
    log_volume_ratio = np.log1p(v / v.shift(1).where(v.shift(1) != 0))
    weighted_move = (price_ratio - 1).abs() * v
    for window in WINDOWS:
        cr, hr, lr, vr = (s.rolling(window, min_periods=1) for s in (c, h, l, v))
        slope, r2, residual = rolling_regression(c, window)
        highest, lowest = hr.max(), lr.min()
        # 和 Qlib np.argmax/argmin 一致；并列极值选首次出现的位置。
        imax = hr.apply(lambda a: a.argmax() + 1, raw=True) / window
        imin = lr.apply(lambda a: a.argmin() + 1, raw=True) / window
        positive = (c > previous).astype(float).rolling(window, min_periods=1).mean()
        negative = (c < previous).astype(float).rolling(window, min_periods=1).mean()
        sump = price_delta.clip(lower=0).rolling(window, min_periods=1).sum()
        sumn = (-price_delta).clip(lower=0).rolling(window, min_periods=1).sum()
        sumabs = price_delta.abs().rolling(window, min_periods=1).sum() + EPS
        vsump = volume_delta.clip(lower=0).rolling(window, min_periods=1).sum()
        vsumn = (-volume_delta).clip(lower=0).rolling(window, min_periods=1).sum()
        vsumabs = volume_delta.abs().rolling(window, min_periods=1).sum() + EPS
        wr = weighted_move.rolling(window, min_periods=1)
        current = {
            'ROC': c.shift(window)/c, 'MA': cr.mean()/c, 'STD': cr.std(ddof=1)/c,
            'BETA': slope/c, 'RSQR': r2, 'RESI': residual/c,
            'MAX': highest/c, 'MIN': lowest/c, 'QTLU': cr.quantile(.8)/c,
            'QTLD': cr.quantile(.2)/c, 'RANK': cr.rank(method='average', pct=True),
            'RSV': (c-lowest)/(highest-lowest+EPS), 'IMAX': imax, 'IMIN': imin,
            'IMXD': imax-imin, 'CORR': qlib_corr(c, log_volume, window),
            'CORD': qlib_corr(price_ratio, log_volume_ratio, window),
            'CNTP': positive, 'CNTN': negative, 'CNTD': positive-negative,
            'SUMP': sump/sumabs, 'SUMN': sumn/sumabs, 'SUMD': (sump-sumn)/sumabs,
            'VMA': vr.mean()/(v+EPS), 'VSTD': vr.std(ddof=1)/(v+EPS),
            'WVMA': wr.std(ddof=1)/(wr.mean()+EPS),
            'VSUMP': vsump/vsumabs, 'VSUMN': vsumn/vsumabs, 'VSUMD': (vsump-vsumn)/vsumabs,
        }
        out.update({f'{name}{window}': value for name, value in current.items()})
    return pd.DataFrame(out, index=bars.index)[FEATURE_NAMES].replace([np.inf, -np.inf], np.nan)

## 5. 标签与时间切分

`LABEL0`：t+1 至 t+2 的复权收盘收益；`LABEL5`、`LABEL10` 同样从 t+1 起算。
每个标签另存进入和结束日期，不混入特征列。若进入/退出日没有有效成交，标签为 NaN。
原始样本保留缺标签的最新日期，便于预测；有监督训练加载时再去掉缺标签行。

按日期分为 train / valid / test。每个标签独立检查其结束日期是否跨越分段边界，
跨界标签保留原值，但在 `load_model_data` 中剔除，避免训练标签使用验证期价格。
特征可以使用分段起点之前的历史行情，这是当时已知的信息。

In [ ]:
def label_name(horizon):
    return 'LABEL0' if horizon == 1 else f'LABEL{horizon}'

def make_labels(bars, horizons=LABEL_HORIZONS, entry_lag=LABEL_ENTRY_LAG):
    if entry_lag < 1 or any(h < 1 for h in horizons):
        raise ValueError('标签进入延迟和持有期必须为正整数')
    labels = pd.DataFrame(index=bars.index)
    dates = pd.Series(bars.index, index=bars.index)
    entry = bars.close.shift(-entry_lag)
    entry_ok = bars.tradeable_observed.shift(-entry_lag, fill_value=False)
    labels['label_entry_date'] = dates.shift(-entry_lag)
    for horizon in horizons:
        name = label_name(horizon)
        exit_lag = entry_lag + horizon
        exit_price = bars.close.shift(-exit_lag)
        exit_ok = bars.tradeable_observed.shift(-exit_lag, fill_value=False)
        labels[name] = (exit_price / entry - 1).where(entry_ok & exit_ok)
        labels[f'{name}_end_date'] = dates.shift(-exit_lag)
    return labels

def attach_splits(frame, train_end=TRAIN_END, valid_end=VALID_END, test_end=TEST_END):
    train_end, valid_end, test_end = map(pd.Timestamp, (train_end, valid_end, test_end))
    if not train_end < valid_end < test_end:
        raise ValueError('必须满足 TRAIN_END < VALID_END < TEST_END')
    result = frame.copy()
    day = result.index.get_level_values('datetime')
    result['split'] = np.select([day <= train_end, day <= valid_end, day <= test_end],
                                ['train', 'valid', 'test'], default='outside')
    limits = {'train': train_end, 'valid': valid_end, 'test': test_end}
    boundary = pd.to_datetime(result['split'].map(limits))
    for horizon in LABEL_HORIZONS:
        name = label_name(horizon)
        result[f'{name}_usable'] = result[name].notna() & result[f'{name}_end_date'].le(boundary)
    return result

def build_one(raw, calendar, min_history=MIN_HISTORY):
    instrument = raw.ts_code.iloc[0]
    if raw.ts_code.nunique() != 1:
        raise ValueError('build_one 只接受一只股票')
    bars = prepare_instrument(raw, calendar, ADJUST_VOLUME)
    feature = alpha158(bars)
    frame = feature.join(make_labels(bars))
    frame['history_count'] = bars.history_count
    keep = bars.tradeable_observed & bars.history_count.ge(min_history)
    frame = frame.loc[keep].copy()
    frame['instrument'] = instrument
    frame = frame.reset_index().set_index(['datetime', 'instrument']).sort_index()
    frame = attach_splits(frame)
    frame[FEATURE_NAMES] = frame[FEATURE_NAMES].astype('float32')
    return frame

## 6. 公式及数据泄漏自检（无需等待下载完成）

模拟数据仅用于检查：158 列、已知公式、回归结果、复权与 VWAP 单位、拆股连续性、
停牌后的标签索引、边界标签剔除，以及修改未来行情不会改变过去因子。
模拟数据不参与正式导出。另读取已下载的第一个交易日检查真实字段。

In [ ]:
def run_self_checks():
    rng = np.random.default_rng(158)
    dates = pd.bdate_range('2022-01-04', periods=100)
    close = 20*np.exp(np.cumsum(rng.normal(0, .01, len(dates))))
    opening = close * (1 + rng.normal(0, .003, len(dates)))
    volume = rng.integers(1000, 9000, len(dates)).astype(float)
    raw = pd.DataFrame({'ts_code': 'TEST.SZ', 'trade_date': dates,
                        'open': opening, 'high': np.maximum(opening, close)*1.01,
                        'low': np.minimum(opening, close)*.99, 'close': close,
                        'vol': volume, 'amount': close*volume/10, 'adj_factor': 2.0})
    bars = prepare_instrument(raw, dates)
    factors = alpha158(bars)
    assert factors.shape == (100, 158) and list(factors) == FEATURE_NAMES
    np.testing.assert_allclose(bars.vwap, close)
    np.testing.assert_allclose(factors.VWAP0, 1)
    np.testing.assert_allclose(factors.ROC5.iloc[5:], close[:-5]/close[5:])
    np.testing.assert_allclose(factors.MA5.iloc[-1], close[-5:].mean()/close[-1])
    np.testing.assert_allclose(factors.STD5.iloc[-1], close[-5:].std(ddof=1)/close[-1])
    slope, intercept = np.polyfit(np.arange(5), close[-5:], 1)
    np.testing.assert_allclose(factors.BETA5.iloc[-1], slope/close[-1], atol=1e-12)
    np.testing.assert_allclose(factors.RESI5.iloc[-1], (close[-1]-(intercept+4*slope))/close[-1], atol=1e-12)
    np.testing.assert_allclose(factors.RSQR5.iloc[-1], np.corrcoef(np.arange(5), close[-5:])[0,1]**2)
    labels = make_labels(bars)
    np.testing.assert_allclose(labels.LABEL0.iloc[:-2], close[2:]/close[1:-1]-1)
    assert labels.LABEL0.tail(2).isna().all()
    changed = raw.copy()
    changed.loc[70:, ['open', 'high', 'low', 'close', 'amount']] *= 3
    changed.loc[70:, 'vol'] *= 2
    pd.testing.assert_frame_equal(factors.iloc[:70], alpha158(prepare_instrument(changed, dates)).iloc[:70])
    # 2:1 拆股：价格减半、股数翻倍、因子翻倍，复权后应连续。
    split = raw.copy()
    split.loc[50:, ['open', 'high', 'low', 'close']] /= 2
    split.loc[50:, 'vol'] *= 2
    split.loc[50:, 'adj_factor'] *= 2
    adjusted = prepare_instrument(split, dates)
    np.testing.assert_allclose(adjusted.close, bars.close)
    np.testing.assert_allclose(adjusted.volume, bars.volume)
    np.testing.assert_allclose(adjusted.vwap, bars.vwap)
    suspended = prepare_instrument(raw.drop(index=50), dates)
    assert make_labels(suspended).LABEL0.iloc[48:50].isna().all()
    flat = bars.copy()
    flat[['open', 'high', 'low', 'close', 'vwap']] = 10.
    flat['volume'] = 0.
    constant = alpha158(flat)
    assert constant.CORR5.isna().all() and constant.RSQR5.isna().all()
    assert not np.isinf(constant.to_numpy()).any()
    np.testing.assert_allclose(constant.IMAX5.iloc[4:], 1/5)
    np.testing.assert_allclose(constant.RANK5.iloc[4:], .6)
    sample = build_one(raw, dates)
    assert len(sample) == 40 and sample.index.names == ['datetime', 'instrument']
    purged = attach_splits(sample, dates[80], dates[90], dates[-1])
    last_train = purged[purged['split'].eq('train')].tail(2)
    assert not last_train.LABEL0_usable.any()
    assert (purged.loc[purged.LABEL0_usable & purged['split'].eq('train'), 'LABEL0_end_date'] <= dates[80]).all()
    print('通过：158 列、收益标签、回归、VWAP、复权、停牌、常数序列、未来扰动和跨分段标签检查。')
    return {'checks': 'passed', 'rows': len(raw), 'features': len(FEATURE_NAMES)}

self_check_result = run_self_checks()
# 仅读取一个已有日期，不等待、不调用接口，也不读取下载 notebook 中的凭据。
available = sorted(set(manifests['daily']) & set(manifests['adj_factor']))
if available:
    first = available[0]
    one_day = {'daily': {first: manifests['daily'][first]},
               'adj_factor': {first: manifests['adj_factor'][first]}}
    real_sample = load_daily_panel(one_day)
    first_stock = real_sample[real_sample.ts_code.eq(real_sample.ts_code.iloc[0])]
    real_bars = prepare_instrument(first_stock, pd.DatetimeIndex([pd.Timestamp(first)]))
    real_features = alpha158(real_bars)
    assert real_features.shape[1] == 158
    display(real_features[['KMID', 'KLEN', 'OPEN0', 'VWAP0']])
    print(f'真实文件字段检查通过：{first}，{len(real_sample):,} 条行情；单日不能验证完整滚动窗口。')

## 7. 全量计算、质量统计及分批保存

每 100 只股票保存一份，降低特征计算峰值内存。读取原始日线仍需容纳所选股票的 OHLCV；
内存较小时先设置 `INSTRUMENTS` 为研究股票列表。
输出保存到 `data/processed/alpha158/run_<时间>_<标识>/`，成功后更新 `latest_run.json`。
失败不会覆盖旧结果；临时输出会清理。不会覆盖、重写原始 CSV。

产物包括 `part-*.parquet` 或 `part-*.csv.gz`、`feature_names.json`、`metadata.json`、
`feature_quality.csv` 和 `split_summary.csv`。
NaN 比例在原始因子层面统计，保留零方差窗口等合理缺失。

输入审计另存 `input_coverage.csv`、`excluded_input_rows.csv`；元数据记录股票池及其哈希、过滤和隔离计数。

In [ ]:
def write_partition(frame, path, output_format):
    if output_format == 'parquet':
        frame.to_parquet(path, index=True)
    else:
        frame.to_csv(path, index=True, compression='gzip')

def build_dataset(raw_dir=RAW_DIR, output_root=OUTPUT_ROOT):
    # 重新取快照：上方预检可能在下载结束前执行过。
    cal, files, report, ready = check_coverage(raw_dir, START_DATE, END_DATE)
    if not ready:
        display(report)
        raise RuntimeError('必需日线或复权数据未下载完整，未构建正式数据集')
    panel = load_daily_panel(files, INSTRUMENTS, audit_dir=raw_dir.parent / 'checks' / 'alpha158_input')
    input_coverage = panel.attrs['input_coverage']
    excluded_input_rows = panel.attrs['excluded_input_rows']
    universe_info = panel.attrs['universe']
    input_quality = panel.attrs['input_quality']
    panel.attrs = {}  # 避免审计对象传播到 groupby 的每个股票分组
    output_root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    run_id = f'run_{stamp}_{uuid.uuid4().hex[:8]}'
    staging = output_root / f'.{run_id}.tmp'
    target = output_root / run_id
    staging.mkdir()
    suffix = '.parquet' if OUTPUT_FORMAT == 'parquet' else '.csv.gz'
    counts = pd.Series(0, index=FEATURE_NAMES, dtype='int64')
    rows = 0
    summaries, buffer = [], []
    partitions = 0
    stock_count = panel.ts_code.nunique()
    try:
        for number, (_, stock) in enumerate(panel.groupby('ts_code', sort=True), 1):
            result = build_one(stock, cal)
            if not result.empty:
                counts += result[FEATURE_NAMES].notna().sum()
                rows += len(result)
                summary = result.groupby('split').agg(rows=('history_count', 'size'),
                                                     labeled=('LABEL0_usable', 'sum'))
                summaries.append(summary)
                buffer.append(result)
            if number % BATCH_SIZE == 0 or number == stock_count:
                if buffer:
                    part = pd.concat(buffer).sort_index()
                    write_partition(part, staging / f'part-{partitions:05d}{suffix}', OUTPUT_FORMAT)
                    partitions += 1
                    buffer.clear()
                print(f'完成股票 {number}/{stock_count}，累计样本 {rows:,}')
        if rows == 0:
            raise ValueError('预热和有效成交过滤后没有样本；请扩大日期范围或检查股票范围')
        quality = pd.DataFrame({'non_null': counts, 'missing_rate': 1-counts/rows})
        quality.index.name = 'feature'
        quality.to_csv(staging / 'feature_quality.csv')
        input_coverage.to_csv(staging / 'input_coverage.csv', index=False)
        excluded_input_rows.to_csv(staging / 'excluded_input_rows.csv', index=False)
        split_summary = pd.concat(summaries).groupby(level=0).sum()
        split_summary.to_csv(staging / 'split_summary.csv')
        metadata = {
            'created_utc': stamp, 'rows': rows, 'instruments': stock_count,
            'features': len(FEATURE_NAMES), 'partitions': partitions, 'format': OUTPUT_FORMAT,
            'universe': universe_info, 'input_quality': input_quality,
            'start_date': START_DATE, 'end_date': END_DATE, 'min_history': MIN_HISTORY,
            'train_end': TRAIN_END, 'valid_end': VALID_END, 'test_end': TEST_END,
            'label_horizons': list(LABEL_HORIZONS), 'label_entry_lag': LABEL_ENTRY_LAG,
            'adjust_volume': ADJUST_VOLUME, 'volume_unit': 'shares',
            'price_adjustment': 'adj_factor / first_observed_adj_factor',
            'label_formula': 'close[t+entry_lag+horizon] / close[t+entry_lag] - 1',
            'source': 'https://github.com/microsoft/qlib/blob/main/qlib/contrib/data/loader.py',
            'pandas': pd.__version__, 'numpy': np.__version__,
            'source_files': {group: [str(path.relative_to(raw_dir)) for path in mapping.values()]
                             for group, mapping in files.items()},
        }
        (staging / 'feature_names.json').write_text(json.dumps(FEATURE_NAMES, indent=2), encoding='utf-8')
        (staging / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
        staging.rename(target)
        pointer = output_root / 'latest_run.json.tmp'
        pointer.write_text(json.dumps({'run_dir': target.name}), encoding='utf-8')
        pointer.replace(output_root / 'latest_run.json')
    except Exception:
        # 仅删除此函数本次创建的临时目录；路径固定在 output_root 内。
        if staging.is_dir() and staging.resolve().parent == output_root.resolve():
            shutil.rmtree(staging)
        raise
    display(split_summary)
    display(quality.sort_values('missing_rate', ascending=False).head(10))
    print(f'已保存：{target}')
    return target

run_dir = None
if RUN_BUILD:
    # 每次运行都重新检查，允许下载在编辑 notebook 期间完成。
    calendar, manifests, coverage, data_ready = check_coverage(RAW_DIR, START_DATE, END_DATE)
    if data_ready:
        run_dir = build_dataset()
    else:
        display(coverage)
        print('正式构建已跳过：数据仍在下载。下载结束后从第 2 节重新运行即可。')
else:
    print('RUN_BUILD=False：仅做预检和自检。')

## 8. 后续模型的数据接口

`load_model_data(split='train', label='LABEL0')` 返回 `(X, y)`，索引为 `(datetime, instrument)`。
X 仅有 158 列，不包含收益、复权因子、日期切分标记或未来日期。
加载函数默认保留特征 NaN；LightGBM 等模型可原生处理缺失。
对需要完整数值的模型，使用下方 `fit_preprocessor` / `transform_features`，仅用训练集拟合中位数和均值方差。
全为空或常数训练列也能保留原有 158 列结构。
`load_model_data` 会将所选分段读入内存；大范围训练可按 `iter_partitions` 分批读取。

In [ ]:
def resolve_run(run=None):
    if run is not None:
        return Path(run)
    pointer = OUTPUT_ROOT / 'latest_run.json'
    if not pointer.exists():
        raise FileNotFoundError('尚无成功构建的数据集，请先完成第 7 节')
    return OUTPUT_ROOT / json.loads(pointer.read_text(encoding='utf-8'))['run_dir']

def iter_partitions(run=None):
    folder = resolve_run(run)
    metadata = json.loads((folder / 'metadata.json').read_text(encoding='utf-8'))
    pattern = 'part-*.parquet' if metadata['format'] == 'parquet' else 'part-*.csv.gz'
    paths = sorted(folder.glob(pattern))
    if not paths:
        raise FileNotFoundError(f'无数据分区：{folder}')
    for path in paths:
        if metadata['format'] == 'parquet':
            yield pd.read_parquet(path)
        else:
            frame = pd.read_csv(path, dtype={'instrument': str}, parse_dates=['datetime'])
            for field in frame.columns:
                if field == 'label_entry_date' or field.endswith('_end_date'):
                    frame[field] = pd.to_datetime(frame[field])
            yield frame.set_index(['datetime', 'instrument'])

def load_model_data(split='train', label='LABEL0', run=None, instruments=None):
    if split not in ('train', 'valid', 'test'):
        raise ValueError('split 必须为 train、valid 或 test')
    chunks = []
    for frame in iter_partitions(run):
        usable = f'{label}_usable'
        if usable not in frame:
            raise KeyError(f'数据集不包含标签 {label}')
        mask = frame['split'].eq(split) & frame[usable].eq(True)
        if instruments is not None:
            mask &= frame.index.get_level_values('instrument').isin(instruments)
        chunks.append(frame.loc[mask, FEATURE_NAMES + [label]])
    data = pd.concat(chunks).sort_index()
    if data.empty:
        raise ValueError(f'{split}/{label} 没有可用样本，请检查日期、预热及标签边界')
    return data[FEATURE_NAMES].astype('float32'), data[label].astype('float32')

def fit_preprocessor(X_train):
    # 调用方必须传入训练分段；不读验证和测试数据。
    median = X_train.median().fillna(0.)
    filled = X_train.fillna(median)
    mean = filled.mean()
    std = filled.std(ddof=0).replace(0., 1.).fillna(1.)
    return {'columns': list(X_train), 'median': median, 'mean': mean, 'std': std}

def transform_features(X, state):
    if list(X) != state['columns']:
        raise ValueError('特征列或顺序与训练集不一致')
    return ((X.fillna(state['median']) - state['mean']) / state['std']).astype('float32')

# 数据构建成功后按需取消注释。此处不自动加载全市场或训练模型。
# X_train, y_train = load_model_data('train', run=run_dir)
# X_valid, y_valid = load_model_data('valid', run=run_dir)
# X_test, y_test = load_model_data('test', run=run_dir)
# state = fit_preprocessor(X_train)
# X_train = transform_features(X_train, state)
# X_valid = transform_features(X_valid, state)
# X_test = transform_features(X_test, state)

## 运行结果如何判断

- 自检应打印“通过”，并展示已下载首日的 4 个示例因子。
- 缺少日期时，覆盖表会列出 `missing_days`，正式构建不产出不完整数据集。
- 下载完成且构建成功后，输出路径、样本数、分段标签数与缺失率会显示在第 7 节。
- 某分段标签数为 0 时，需要补齐日期或调整时间切分；不随机切分、不用测试数据拟合预处理。
- 运行全量前应确认下载进程已结束；文件覆盖齐全只代表文件存在，内容校验在正式读取时执行。

标准 Alpha158 不要求 `daily_basic`、行业、财务、指数或筹码数据。
若以后加入这些扩展因子，请单独命名并保存，避免改变各模型共用的标准 158 列。